In [26]:
import re
import pandas as pandas
import numpy as numpy
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

import re
import pandas as pd
from googleapiclient.discovery import build
import dotenv
dotenv.load_dotenv()



True

In [27]:
API_KEY = dotenv.get_key(".env", "api")
MAX_COMMENTS = 300


def extract_video_id(url):
    match = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11})", url)
    return match.group(1) if match else None


In [ ]:

def fetch_comments(video_id):
    youtube = build("youtube", "v3", developerKey=API_KEY)
    comments = []

    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )

    while request and len(comments) < MAX_COMMENTS:
        response = request.execute()

        for item in response["items"]:
            comments.append(
                item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
            )

        request = youtube.commentThreads().list_next(request, response)

    return comments[:MAX_COMMENTS]



In [ ]:
video_url = input("Enter YouTube video URL: ")
video_id = extract_video_id(video_url)

if not video_id:
    print("Invalid YouTube URL")
    exit()

comments = fetch_comments(video_id)

pd.DataFrame(comments, columns=["comment_text"]).to_csv(
    "comments.csv", index=False
)

print(f"Saved {len(comments)} comments to comments.csv")

Invalid YouTube URL


NameError: name 'fetch_comments' is not defined

: 

In [7]:
df = pandas.read_csv("comments.csv") 
df.sample(10)

,comment_text
25,hi sir The Bag of Words (BoW) model does not s...
68,Very nice and very good start point. Can you p...
93,Feature extraction from text / text representa...
129,Best series on NLP
150,Incredibile ❤️
104,2 hours of pure diamond mine.
99,57:00
74,The example you gave for bigrams better than u...
111,"Can you add RNN, LSTMs, and modern NLP using t..."
102,sir please share the one note link


In [16]:
df["length"] = df["comment_text"].astype(str).apply(len)
df.describe()

,length
count,153.000000
mean,86.843137
std,118.901779
min,4.000000
25%,27.000000
50%,50.000000
75%,109.000000
max,1103.000000


In [11]:
def clean_text(text):
    text = str(text).lower()                          # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)        # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)              # remove emojis & symbols
    text = re.sub(r"\s+", " ", text).strip()          # remove extra spaces
    return text


In [12]:
df["clean_comment"] = df["comment_text"].apply(clean_text)
df.head(10)

,comment_text,length,clean_comment
0,"Sir like, comment karna to banta hai. lekin sh...",149,sir like comment karna to banta hai lekin shar...
1,devuduuuuuuuuuuuuuuu sir meeru,30,devuduuuuuuuuuuuuuuu sir meeru
2,Nitish sir all lectures are amazing,35,nitish sir all lectures are amazing
3,Full informative video❤ no extra talk smooth a...,149,full informative video no extra talk smooth an...
4,great video sir,15,great video sir
5,Loved it 😀,10,loved it
6,"I was scared to death about ""NLP"" , after watc...",145,i was scared to death about nlp after watching...
7,Simply an amazing course. Thank you so much sir.,48,simply an amazing course thank you so much sir
8,"As far as i can thing,, they use laplace smoot...",262,as far as i can thing they use laplace smoothi...
9,damn best resource ⚒,20,damn best resource


In [14]:
df = df[df["clean_comment"].str.len() > 0]
df["clean_comment"].sample(10, random_state=42)


91                    share the link for collab notebook
93     feature extraction from text text representati...
108                     sir please continue the playlist
126         for me you are the best data science teacher
33                                           great class
125    at bow doesnt consider the sequence of sentenc...
84     sir i dont understand for idea of tf idf at si...
87     liked and shared your video subscribed your ch...
20     where are jupyter notebook of this session and...
16                                       love your class
Name: clean_comment, dtype: object

In [17]:
nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Gravity\AppData\Roaming\nltk_data...


In [18]:
test_comment = df["clean_comment"].iloc[0]
sia.polarity_scores(test_comment)

{'neg': 0.046, 'neu': 0.786, 'pos': 0.168, 'compound': 0.5267}

In [22]:
def get_sentiment(text):
    score = sia.polarity_scores(text)["compound"]
    if score >= 0.04:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

In [23]:
df["sentiment"] = df["clean_comment"].apply(get_sentiment)
df.head(10)

,comment_text,length,clean_comment,sentiment
0,"Sir like, comment karna to banta hai. lekin sh...",149,sir like comment karna to banta hai lekin shar...,positive
1,devuduuuuuuuuuuuuuuu sir meeru,30,devuduuuuuuuuuuuuuuu sir meeru,neutral
2,Nitish sir all lectures are amazing,35,nitish sir all lectures are amazing,positive
3,Full informative video❤ no extra talk smooth a...,149,full informative video no extra talk smooth an...,positive
4,great video sir,15,great video sir,positive
5,Loved it 😀,10,loved it,positive
6,"I was scared to death about ""NLP"" , after watc...",145,i was scared to death about nlp after watching...,negative
7,Simply an amazing course. Thank you so much sir.,48,simply an amazing course thank you so much sir,positive
8,"As far as i can thing,, they use laplace smoot...",262,as far as i can thing they use laplace smoothi...,neutral
9,damn best resource ⚒,20,damn best resource,positive


In [24]:
df["sentiment"].value_counts()


sentiment
positive    120
neutral      25
negative      8
Name: count, dtype: int64

In [25]:
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df["label"] = df["sentiment"].map(label_map)
df[["clean_comment", "sentiment", "label"]].head()


,clean_comment,sentiment,label
0,sir like comment karna to banta hai lekin shar...,positive,2
1,devuduuuuuuuuuuuuuuu sir meeru,neutral,1
2,nitish sir all lectures are amazing,positive,2
3,full informative video no extra talk smooth an...,positive,2
4,great video sir,positive,2
